In [91]:
# import libraries
import os
import ee
import geemap

# Initialize the Earth Engine API
ee.Initialize()

In [ ]:
#create map
Map = geemap.Map()
Map

In [ ]:
# Create roi ee feature collection
roi = ee.FeatureCollection(Map.draw_last_feature)

In [ ]:
# Create median composite
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
)

image = s2_collection.median().clip(roi)

In [ ]:
# True Color (Natural Color)
vis_params_truecolor = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 2000,
    'gamma': 1
}

# Add layers to map
Map.addLayer(image, vis_params_truecolor, 'True Color')
Map.centerObject(roi)
Map

In [ ]:
# Calculate NDVI: (NIR - Red) / (NIR + Red)
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Define a color palette for NDVI
ndvi_palette = [ 
    'ffffff', 'ce7e45', 'df923d', 'f1b555', 'fcd163', '99b718', '74a901', '66a000', '529400', 
    '3e8601', '207401', '056201', '004c00', '023b01', '012e01', '011d01', '011301' 
]

# Add NDVI layer to the map
Map.addLayer(ndvi, {'min': 0, 'max': 1, 'palette': ndvi_palette}, 'NDVI')

# Display the map
Map

In [92]:
# Export Image
out_dir = os.path.join(os.path.expanduser("~"), "Downloads")
filename = os.path.join(out_dir, "sentinel2_image.tif")
geemap.ee_export_image(
    ndvi, 
    filename=filename, 
    scale=10, 
    region=roi.geometry()
)

Generating URL ...
Please wait ...
Data downloaded to C:\Users\user\Downloads\sentinel2_image.tif


In [93]:
# Add exported image to map
aprx = arcpy.mp.ArcGISProject("CURRENT")
aprx.activeMap.addDataFromPath(filename)